# 🧠🤖 第3周-Day2：SFT监督微调的关键细节

今天我们来深入理解大模型训练中最关键的一步：**监督微调（SFT）**！

💡 **学习目标**：
- 理解SFT的核心原理和重要性
- 掌握高质量指令数据的制作方法
- 学会使用LoRA进行高效微调
- 了解SFT的评估和验证方法

## 🔄 昨日复习回顾

昨天我们学习了预训练的基础知识：
- 大规模数据（万亿token）是能力涌现的基础
- 模型规模增大→能力突然突破（Scaling Law）
- 预训练成本：算力+数据质量缺一不可

今天我们要学习的SFT，就是在预训练的基础上让模型"学会如何对话"！

## 📚 SFT核心概念详解

### 什么是SFT？

**监督微调（SFT）**：用高质量的指令数据对预训练模型进行微调，让模型学会理解并遵循人类指令。

🔑 **核心原理**：
1. **输入**：高质量指令-回答对
2. **过程**：通过监督学习优化模型参数
3. **输出**：能够理解和遵循指令的对话模型

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'Droid Sans Fallback', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建SFT前后能力对比图
def plot_sft_comparison():
    categories = ['指令理解', '对话连贯性', '知识应用', '安全性', '创造力']
    pre_training = [20, 15, 25, 10, 18]  # 预训练能力分数
    post_sft = [85, 78, 70, 88, 65]      # SFT后能力分数
    
    x = np.arange(len(categories))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, pre_training, width, label='预训练', alpha=0.8, color='lightblue')
    bars2 = ax.bar(x + width/2, post_sft, width, label='SFT后', alpha=0.8, color='lightcoral')
    
    ax.set_ylabel('能力分数 (%)')
    ax.set_title('SFT前后模型能力对比', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 在柱子上方添加数值
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

plot_sft_comparison()

## 🔑 高质量数据制作

### 数据质量 > 数据量

📊 **关键发现**：
- 10条优质指令胜过100条普通数据
- 数据多样性比数量更重要
- 指令设计决定了模型能力的上限

In [ ]:
# 模拟数据质量与效果的关系
def plot_data_quality_impact():
    # 模拟不同数据质量和数量的组合
    data_qualities = [0.1, 0.3, 0.5, 0.7, 0.9]  # 数据质量分数
    data_quantities = [10, 50, 100, 500, 1000]   # 数据数量
    
    # 计算效果分数（假设效果 = 数据质量 * log(数据数量)）
    results = []
    for quality in data_qualities:
        for quantity in data_quantities:
            effectiveness = quality * np.log10(quantity + 1)
            results.append({
                'quality': quality,
                'quantity': quantity,
                'effectiveness': effectiveness
            })
    
    df = pd.DataFrame(results)
    
    # 创建热力图
    pivot_table = df.pivot('quality', 'quantity', 'effectiveness')
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlOrRd', 
                cbar_kws={'label': '效果分数'})
    plt.title('数据质量×数量对SFT效果的影响', fontsize=14, fontweight='bold')
    plt.xlabel('数据数量')
    plt.ylabel('数据质量')
    plt.tight_layout()
    plt.show()

plot_data_quality_impact()

## 🎯 Instruction Template设计

### 指令模板是灵魂

好的指令模板能让模型：
- 明确理解任务边界
- 保持一致的输出格式
- 减少误解和幻觉

📝 **经典模板结构**：
```python
instruction_template = """
任务：{task}
背景：{context}
要求：{requirements}
输出格式：{format}
"""
```

In [ ]:
# 模拟不同指令模板的效果对比
def plot_template_comparison():
    templates = [
        '简单指令',
        '带背景说明',
        '明确格式要求',
        '多步骤指导',
        '角色扮演式'
    ]
    
    # 模拟各种指标分数
    understanding = [65, 75, 82, 88, 92]    # 指令理解度
    consistency = [60, 70, 85, 90, 88]       # 输出一致性
    creativity = [70, 68, 72, 80, 95]         # 创造性
    efficiency = [80, 75, 78, 72, 65]        # 执行效率
    
    x = np.arange(len(templates))
    width = 0.2
    
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.bar(x - 1.5*width, understanding, width, label='指令理解度', alpha=0.8)
    ax.bar(x - 0.5*width, consistency, width, label='输出一致性', alpha=0.8)
    ax.bar(x + 0.5*width, creativity, width, label='创造性', alpha=0.8)
    ax.bar(x + 1.5*width, efficiency, width, label='执行效率', alpha=0.8)
    
    ax.set_ylabel('分数 (%)')
    ax.set_title('不同指令模板的效果对比', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(templates, rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()

plot_template_comparison()

## 🔧 LoRA高效微调

### LoRA核心优势

🚀 **为什么选择LoRA？**
- **参数高效**：只训练少量适配器参数
- **内存友好**：GPU内存需求大幅降低
- **能力保留**：不破坏预训练能力
- **易于部署**：可以轻松切换不同适配器

In [ ]:
# LoRA原理可视化
def plot_lora_principle():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 传统微调
    ax1.set_title('传统微调：所有参数都更新', fontsize=14, fontweight='bold')
    
    # 绘制神经网络层
    layers = ['Input', 'Hidden Layer 1', 'Hidden Layer 2', 'Output']
    layer_size = [3, 5, 5, 2]
    
    positions = []
    y_pos = 0
    for i, (layer, size) in enumerate(zip(layers, layer_size)):
        x_pos = np.linspace(0, 10, size)
        positions.append((x_pos, y_pos, layer, size))
        
        # 绘制节点
        for x in x_pos:
            circle = plt.Circle((x, y_pos), 0.2, color='lightblue', ec='blue')
            ax1.add_patch(circle)
            
        # 标注层名
        ax1.text(5, y_pos + 0.5, layer, ha='center', fontweight='bold')
        
        # 绘制连接线（红色表示需要训练）
        if i > 0:
            prev_x, prev_y, _, prev_size = positions[i-1]
            for x in x_pos:
                for px in prev_x:
                    ax1.plot([px, x], [prev_y, y_pos], 'r-', alpha=0.3, linewidth=1)
        
        y_pos -= 1.5
    
    ax1.set_xlim(-1, 11)
    ax1.set_ylim(-6, 1)
    ax1.set_aspect('equal')
    ax1.axis('off')
    
    # LoRA微调
    ax2.set_title('LoRA微调：只训练适配器', fontsize=14, fontweight='bold')
    
    y_pos = 0
    for i, (layer, size) in enumerate(layers[:3]):  # 只显示前3层
        x_pos = np.linspace(0, 10, size)
        
        # 绘制主要网络（灰色，不需要训练）
        for x in x_pos:
            circle = plt.Circle((x, y_pos), 0.2, color='lightgray', ec='gray')
            ax2.add_patch(circle)
        
        # 标注层名
        ax2.text(5, y_pos + 0.5, layer, ha='center', fontweight='bold')
        
        # 绘制主要连接线（灰色）
        if i > 0:
            prev_x, prev_y, _, prev_size = positions[i-1]
            for x in x_pos:
                for px in prev_x:
                    ax2.plot([px, x], [prev_y, y_pos], 'gray', alpha=0.2, linewidth=1)
        
        y_pos -= 1.5
    
    # 添加LoRA适配器
    lora_x = np.linspace(0, 10, 3)
    lora_y = -4.5
    
    for x in lora_x:
        circle = plt.Circle((x, lora_y), 0.2, color='lightcoral', ec='red')
        ax2.add_patch(circle)
    
    ax2.text(5, lora_y + 0.5, 'LoRA\\nAdapter', ha='center', fontweight='bold', color='red')
    
    # 绘制适配器连接（红色，需要训练）
    for x in lora_x:
        for hx in x_pos[:2]:  # 只连接到前两层
            ax2.plot([hx, x], [-1.5, lora_y], 'r-', alpha=0.6, linewidth=2)
            ax2.plot([hx, x], [-3, lora_y], 'r-', alpha=0.6, linewidth=2)
    
    ax2.set_xlim(-1, 11)
    ax2.set_ylim(-6, 1)
    ax2.set_aspect('equal')
    ax2.axis('off')
    
    # 添加图例
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightblue', label='预训练参数（不更新）'),
        Patch(facecolor='lightcoral', label='LoRA适配器（训练）'),
        Patch(facecolor='gray', label='主要连接（不更新）'),
        Patch(facecolor='red', label='适配器连接（更新）')
    ]
    
    ax2.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.15, 1))
    
    plt.tight_layout()
    plt.show()

plot_lora_principle()

## 📊 SFT评估方法

### 多维度评估体系

📈 **常用评估指标**：
- **ROUGE**：文本摘要质量评估
- **BLEU**：机器翻译质量评估
- **人工评分**：指令遵循度、连贯性、安全性
- **基准测试**：MMLU、HumanEval等

🔍 **过拟合检测**：
- 训练集vs验证集性能差距
- 在不同数据集上的泛化能力
- 人工盲测对比

In [ ]:
# 模拟SFT训练过程中的过拟合现象
def plot_overfitting_detection():
    epochs = list(range(1, 21))
    
    # 训练集损失（持续下降）
    train_loss = [2.5 - 0.1*i + 0.001*i*i for i in epochs]
    
    # 验证集损失（先下降后上升）
    val_loss = [2.3 - 0.12*i for i in range(1, 11)] + \
               [1.1 + 0.05*(i-10) for i in range(11, 21)]
    
    # 准确率
    train_acc = [0.6 + 0.02*i - 0.001*i*i for i in epochs]
    val_acc = [0.65 + 0.025*i for i in range(1, 11)] + \
             [0.9 - 0.01*(i-10) for i in range(11, 21)]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 损失曲线
    ax1.plot(epochs, train_loss, 'b-', label='训练集损失', linewidth=2)
    ax1.plot(epochs, val_loss, 'r-', label='验证集损失', linewidth=2)
    ax1.axvline(x=10, color='gray', linestyle='--', alpha=0.7, label='过拟合开始点')
    ax1.set_xlabel('训练轮数')
    ax1.set_ylabel('损失值')
    ax1.set_title('SFT训练中的过拟合检测', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 准确率曲线
    ax2.plot(epochs, train_acc, 'b-', label='训练集准确率', linewidth=2)
    ax2.plot(epochs, val_acc, 'r-', label='验证集准确率', linewidth=2)
    ax2.axvline(x=10, color='gray', linestyle='--', alpha=0.7, label='过拟合开始点')
    ax2.set_xlabel('训练轮数')
    ax2.set_ylabel('准确率')
    ax2.set_title('准确率变化趋势', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 设置y轴范围
    ax2.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()

    # 显示最优停止点
    print(f"最优停止轮数: 第10轮")
    print(f"此时验证准确率: {val_acc[9]:.3f}")
    print(f"继续训练会导致过拟合！")

plot_overfitting_detection()

## 💡 业务关联思考

### SFT在糖水店AI助手中的应用

🍮 **实际场景**：
- **菜单理解**：让AI理解"招牌红豆冰"的具体制作要求
- **库存管理**：通过指令查询原材料库存
- **客户服务**：处理预订、咨询、投诉等对话
- **销售分析**：分析销售数据并提供建议

🏪 **落地挑战**：
- 如何获取高质量的行业对话数据
- 如何在小数据集上避免过拟合
- 如何处理多语言（普通话、方言）需求
- 如何在低成本硬件上运行微调后的模型

In [ ]:
# 模拟不同行业的SFT数据需求分析
def analyze_industry_sft_needs():
    industries = ['糖水店', '餐厅', '客服中心', '教育', '医疗']
    
    # 各行业SFT关键指标权重（0-1）
    data_needs = [0.8, 0.7, 0.9, 0.6, 0.8]  # 数据需求量
    accuracy_needs = [0.7, 0.8, 0.9, 0.85, 0.95]  # 准确性要求
    speed_needs = [0.6, 0.7, 0.8, 0.5, 0.7]  # 响应速度要求
    cost_needs = [0.9, 0.7, 0.6, 0.8, 0.5]  # 成本敏感度
    
    x = np.arange(len(industries))
    width = 0.15
    
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.bar(x - 1.5*width, data_needs, width, label='数据需求', alpha=0.8, color='skyblue')
    ax.bar(x - 0.5*width, accuracy_needs, width, label='准确性要求', alpha=0.8, color='lightcoral')
    ax.bar(x + 0.5*width, speed_needs, width, label='响应速度', alpha=0.8, color='lightgreen')
    ax.bar(x + 1.5*width, cost_needs, width, label='成本敏感度', alpha=0.8, color='lightyellow')
    
    ax.set_ylabel('重要性权重')
    ax.set_title('各行业SFT需求分析', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(industries)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    
    # 突出显示糖水店的需求
    for i, bar in enumerate(ax.patches[:4]):
        if i < 4:  # 只给糖水店的柱子上加标识
            rect_height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., rect_height + 0.02,
                   f'{rect_needs[i]:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # 添加糖水店特殊标识
    ax.text(-0.375, 1.05, '🍮 糖水店特点：数据丰富但准确性要求中等', 
            transform=ax.transData, ha='center', va='top', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    plt.tight_layout()
    plt.show()

analyze_industry_sft_needs()

## ✏️ 课堂练习（5分钟）

### 快速思考题

❶ SFT和预训练的核心区别是什么？
- **提示**：从目标、数据、方法三个维度思考

❷ 为什么SFT数据质量比数量更重要？
- **提示**：考虑模型的记忆能力和理解能力

❸ LoRA在SFT中的主要优势是什么？
- **提示**：从资源需求和效果保留两方面思考

❹ 如何判断SFT是否过拟合？
- **提示**：观察训练集和验证集的表现差异

❺ SFT后模型会出现哪些新能力？
- **提示**：对比预训练模型和对话模型的特点

**请思考后回复答案，我会帮你批改！**

## 🎯 今日学习总结

### 关键知识点回顾

🎯 **SFT核心原理**：
- 用高质量指令数据让模型学会对话
- 数据质量 > 数据数量
- Instruction Template设计至关重要

🛠️ **技术要点**：
- LoRA参数高效微调
- 多维度评估体系
- 过拟合检测与避免

📊 **效果提升**：
- 指令理解能力显著提升
- 对话连贯性大幅改善
- 安全性和可控性增强

### 明日预告

明天我们将学习**RLHF：人类反馈强化学习详解**，这是让模型更符合人类偏好的关键技术！

🚀 **准备内容**：
- 人类反馈信号收集
- 强化学习基础回顾
- RLHF算法详解
- 与SFT的对比分析

In [ ]:
# 学习进度跟踪
def plot_learning_progress():
    weeks = ['W1', 'W2', 'W3']
    progress = [75, 85, 60]  # 各周完成度
    
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(weeks, progress, color=['lightblue', 'lightgreen', 'lightcoral'])
    
    ax.set_ylabel('完成度 (%)')
    ax.set_title('第1-3周学习进度', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 100)
    
    # 在柱子上添加进度条
    for bar, p in zip(bars, progress):
        # 背景条
        bg_bar = plt.Rectangle((bar.get_x() - 0.1, 0), bar.get_width() + 0.2, 100, 
                             facecolor='lightgray', alpha=0.3)
        ax.add_patch(bg_bar)
        
        # 进度条
        progress_bar = plt.Rectangle((bar.get_x() - 0.1, 0), bar.get_width() + 0.2, p, 
                                   facecolor=bar.get_facecolor(), alpha=0.8)
        ax.add_patch(progress_bar)
        
        # 百分比文字
        ax.text(bar.get_x() + bar.get_width()/2, p + 2, f'{p}%', 
               ha='center', va='bottom', fontweight='bold')
    
    # 添加当前进度指示器
    ax.axhline(y=60, color='red', linestyle='--', alpha=0.7, label='当前进度')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_learning_progress()

print("🎉 恭喜完成第3周-Day2学习！")
print("📚 明天继续：RLHF人类反馈强化学习详解")
print("💪 保持节奏，我们正在稳步前进！")